In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shai-music-genre-classification/test (2).csv
/kaggle/input/competitions/shai-music-genre-classification/sample_submission (2).csv
/kaggle/input/competitions/shai-music-genre-classification/train (1).csv


In [2]:
import pandas as pd
import numpy as np

# Đường dẫn mặc định của Kaggle cho bài toán này
path_train = '../input/competitions/shai-music-genre-classification/train (1).csv'
path_test = '../input/competitions/shai-music-genre-classification/test (2).csv'
path_sub = '../input/competitions/shai-music-genre-classification/sample_submission (2).csv'

# Đọc file CSV lên thành DataFrame
df_train = pd.read_csv(path_train)
df_test = pd.read_csv(path_test)
df_sub = pd.read_csv(path_sub)

# In ra danh sách tất cả các cột để chúng ta kiểm tra
print("Danh sách các cột trong tập Train:")
print(df_train.columns.tolist())


Danh sách các cột trong tập Train:
['Id', 'Artist Name', 'Track Name', 'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature', 'Class']


In [3]:
print("\nDanh sách 5 dòng đầu tiên của tập Train:")
display(df_train.head(5))


Danh sách 5 dòng đầu tiên của tập Train:


,Id,Artist Name,Track Name,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,Class
0,1,Marina Maximilian,Not Afraid,37.0,0.334,0.536,9.0,-6.649,0,0.0381,0.378000,NaN,0.106,0.235,152.429,204947.0,4,9
1,2,The Black Keys,Howlin' for You,67.0,0.725,0.747,11.0,-5.545,1,0.0876,0.027200,0.0468,0.104,0.380,132.921,191956.0,4,6
2,3,Royal & the Serpent,phuck u,NaN,0.584,0.804,7.0,-6.094,1,0.0619,0.000968,0.6350,0.284,0.635,159.953,161037.0,4,10
3,4,Detroit Blues Band,Missing You,12.0,0.515,0.308,NaN,-14.711,1,0.0312,0.907000,0.0213,0.300,0.501,172.472,298093.0,3,2
4,5,Coast Contra,My Lady,48.0,0.565,0.777,6.0,-5.096,0,0.2490,0.183000,NaN,0.211,0.619,88.311,254145.0,4,5


In [4]:
# KHAI BÁO TÊN CỘT CHÍNH XÁC THEO DỮ LIỆU CỦA BẠN
ten_cot_nhan = 'Class'  # Cột nhãn cần dự đoán
ten_cot_id = 'Id'       # Cột ID của bài toán

# 1. Tách tập Train thành X (các đặc trưng để học) và y (nhãn kết quả)
# Chúng ta bỏ cột 'Class' (nhãn) và cột 'Id' (vì ID không giúp ích cho việc dự đoán)
X = df_train.drop(columns=[ten_cot_nhan, ten_cot_id])
y = df_train[ten_cot_nhan]

# 2. Chuẩn bị tập Test để dự đoán
# Giữ lại cột 'Id' của tập test để lát nữa ghép vào file nộp bài
test_ids = df_test[ten_cot_id]

# Loại bỏ cột 'Id' ra khỏi tập test trước khi đưa vào mô hình dự đoán
test_X = df_test.drop(columns=[ten_cot_id])

# Đảm bảo tập test có đúng và đủ các cột đặc trưng như tập train
test_X = test_X[X.columns]

# 3. In kết quả ra màn hình để kiểm tra
print("Kích thước X (các đặc trưng huấn luyện):", X.shape)
print("Kích thước y (nhãn huấn luyện):", y.shape)
print("Kích thước test_X (đặc trưng để test):", test_X.shape)

print("\nCác cột sẽ dùng làm đặc trưng đầu vào (X):")
print(X.columns.tolist())

Kích thước X (các đặc trưng huấn luyện): (14396, 16)
Kích thước y (nhãn huấn luyện): (14396,)
Kích thước test_X (đặc trưng để test): (3600, 16)

Các cột sẽ dùng làm đặc trưng đầu vào (X):
['Artist Name', 'Track Name', 'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature']


In [5]:
from sklearn.preprocessing import LabelEncoder

# 1. Tạm thời chỉ lấy các cột có kiểu dữ liệu dạng số (bỏ qua 'Artist Name' và 'Track Name')
# Hoặc chúng ta có thể loại bỏ trực tiếp 2 cột đó:
cot_bo_qua = ['Artist Name', 'Track Name']

X_num = X.drop(columns=cot_bo_qua)
test_X_num = test_X.drop(columns=cot_bo_qua)

print("Các cột đặc trưng dạng số được giữ lại để huấn luyện:")
print(X_num.columns.tolist())

# 2. Mã hóa nhãn (y) từ dạng chữ (tên thể loại nhạc) sang dạng số nguyên
bo_ma_hoa_nhan = LabelEncoder()
y_encoded = bo_ma_hoa_nhan.fit_transform(y)

print("\nNhãn sau khi mã hóa (10 giá trị đầu tiên):")
print(y_encoded[:10])

Các cột đặc trưng dạng số được giữ lại để huấn luyện:
['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature']

Nhãn sau khi mã hóa (10 giá trị đầu tiên):
[ 9  6 10  2  5 10  0  8 10  5]
